# ChatGPT Handoffs

Exports self-contained script or visual-planning packets, then parses the
pasted responses back into the same canonical contracts used by the apps.
This notebook does not generate voice, assets, scene clips, or final video.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if not (PROJECT_ROOT / "src" / "pipeline").exists():
    raise RuntimeError("Run this notebook from the repository root or output directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.orchestration import load_run_context, resolve_voiceover_package

OUTPUT_ROOT = PROJECT_ROOT / "output"
PIPELINE_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline" / "config" / "workbench.yaml"
CACHE_ROOT = PROJECT_ROOT / ".cache" / "ai_content_creation" / "models"
FIXTURE_DIR = PROJECT_ROOT / "tests" / "fixtures" / "media_roundup_reference"
IST = timezone(timedelta(hours=5, minutes=30))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return value


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_ROOT": str(OUTPUT_ROOT),
        "CACHE_ROOT": str(CACHE_ROOT),
        "RUN_ID": RUN_ID,
    }
    if isinstance(value, str):
        return value.format(**context)
    if isinstance(value, list):
        return [render_config_templates(item) for item in value]
    if isinstance(value, dict):
        return {key: render_config_templates(item) for key, item in value.items()}
    return value


In [ ]:
# INPUT_MODE: "run" uses output/runs; "fixture" uses the committed media fixture.
INPUT_MODE = "run"
RUN_ID_OR_CURRENT = "current"
VOICEOVER_VARIANT = None  # Folder name/path; required only when multiple voiceovers match this ChatGPT script.

if INPUT_MODE == "fixture":
    RUN_ID = "fixture_media_roundup_reference"
    RUN_DIR = OUTPUT_ROOT / "tests" / "media_roundup_reference"
    VIDEO_PACKAGING_RESULT_PATH = FIXTURE_DIR / "video_packaging_result.json"
    RUN_PROFILE = yaml.safe_load(PIPELINE_CONFIG_PATH.read_text(encoding="utf-8"))["default_run_profile"]
    PROFILE_CONFIG = deepcopy(yaml.safe_load(PIPELINE_CONFIG_PATH.read_text(encoding="utf-8"))["profiles"][RUN_PROFILE])
    RUN_CONTEXT = {"run_id": RUN_ID, "run_profile": RUN_PROFILE, "profile_config": PROFILE_CONFIG, "run_dir": str(RUN_DIR)}
else:
    RUN_CONTEXT = load_run_context(output_root=OUTPUT_ROOT, run_id=RUN_ID_OR_CURRENT)
    RUN_ID = RUN_CONTEXT["run_id"]
    RUN_DIR = Path(RUN_CONTEXT["run_dir"])
    RUN_PROFILE = RUN_CONTEXT["run_profile"]
    PROFILE_CONFIG = deepcopy(RUN_CONTEXT["profile_config"])
    VIDEO_PACKAGING_RESULT_PATH = RUN_DIR / "video_packaging" / "video_packaging_result.json"

HANDOFF_DIR = RUN_DIR / "handoffs" / "chatgpt"
SCRIPT_RESPONSE_PATH = HANDOFF_DIR / "script" / "chatgpt_response.txt"
SCRIPT_OUTPUT_DIR = RUN_DIR / "script_generation" / "chatgpt"
SCRIPT_RESULT_PATH = SCRIPT_OUTPUT_DIR / "script_generation_result.json"
FULL_SCRIPT_PATH = SCRIPT_OUTPUT_DIR / "full_script.txt"
VISUAL_RESPONSE_PATH = HANDOFF_DIR / "visuals" / "chatgpt_visual_plan_response.txt"
VISUAL_PLAN_OUTPUT_DIR = RUN_DIR / "visuals" / "planning" / "chatgpt"


def optional_summary(name: str) -> dict:
    path = RUN_DIR / name / "summary.json"
    return load_json(path) if path.exists() else {}


def compact_voiceover_timing(package: dict) -> dict:
    pkg = package.get("data", {}).get("voiceover_package") if isinstance(package.get("data"), dict) else None
    if not isinstance(pkg, dict):
        pkg = package.get("voiceover_package") if isinstance(package.get("voiceover_package"), dict) else package
    audio = pkg.get("audio", {}) if isinstance(pkg.get("audio"), dict) else {}
    chunks = pkg.get("chunks", []) if isinstance(pkg.get("chunks"), list) else []
    audio_units = pkg.get("audio_units", []) if isinstance(pkg.get("audio_units"), list) else []
    return {
        "duration_seconds": audio.get("duration_seconds") or pkg.get("duration_seconds"),
        "audio_path": audio.get("full_audio_path") or audio.get("audio_path") or pkg.get("audio_path"),
        "chunk_count": len(chunks),
        "audio_unit_count": len(audio_units),
        "provenance": pkg.get("provenance", {}),
        "audio_units": audio_units,
        "chunks": [{key: chunk.get(key) for key in ("chunk_id", "order", "start_seconds", "end_seconds", "duration_seconds", "text")} for chunk in chunks if isinstance(chunk, dict)],
    }


def matching_voiceover_package(script_result: dict) -> tuple[Path, dict]:
    script_fingerprint = script_result["data"]["script_package"]["provenance"]["script_fingerprint"]
    return resolve_voiceover_package(run_dir=RUN_DIR, run_id=RUN_ID, script_fingerprint=script_fingerprint, variant=VOICEOVER_VARIANT)

print(f"Input mode: {INPUT_MODE}")
print(f"Run ID:     {RUN_ID}")
print(f"Run dir:    {RUN_DIR}")


## 1. Load The Selected Video Packet


In [ ]:
video_packaging_result = load_json(VIDEO_PACKAGING_RESULT_PATH)
if video_packaging_result.get("status") != "ok":
    raise RuntimeError(video_packaging_result.get("summary", {}))
video_packet = video_packaging_result["data"]["video_packet"]
print(f"Selected stories: {len(video_packet.get('stories', []))}")
print(f"Duration policy:  {video_packet.get('duration_policy', {}).get('mode', 'content_driven')}")


## 2. Export Script-Writing Handoff


In [ ]:
from src.pipeline.handoffs.chatgpt_exporter import run as run_chatgpt_exporter

script_handoff_dir = HANDOFF_DIR / "script" / "packet"
script_handoff_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {"profile": "content_packet", "prompt_version": "v1", "output_dir": str(script_handoff_dir)},
    "input": {
        "manifest": RUN_CONTEXT,
        "artifacts": {"content_packet": video_packet},
        "summaries": {name: optional_summary(name) for name in ("news_collection", "evidence_prep", "story_mapping", "video_packaging")},
    },
}
script_handoff_result = run_chatgpt_exporter(script_handoff_input)
if script_handoff_result["status"] != "ok":
    raise RuntimeError(script_handoff_result["summary"])
paths = write_handoff_files(script_handoff_result["data"]["handoff_package"]["files"], script_handoff_dir)
write_json(HANDOFF_DIR / "script" / "handoff_input.json", script_handoff_input)
write_json(HANDOFF_DIR / "script" / "handoff_result.json", script_handoff_result)
SCRIPT_RESPONSE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not SCRIPT_RESPONSE_PATH.exists():
    SCRIPT_RESPONSE_PATH.write_text("Paste the full CHATGPT_PIPELINE_RESPONSE_V1 response here, then run section 3.\n", encoding="utf-8")
print("Paste these files into ChatGPT in order:")
for path in paths:
    print(path)
print(f"Response file: {SCRIPT_RESPONSE_PATH}")


## 3. Parse Script Response

Paste the complete response into the printed file. Explicit
`[[SECTION:<section_id>]]` markers preserve exact story boundaries and are
removed before voice generation.


In [ ]:
from src.pipeline.handoffs.chatgpt_script_parser import run as parse_chatgpt_script

response_text = SCRIPT_RESPONSE_PATH.read_text(encoding="utf-8")
parser_config = PROFILE_CONFIG.get("script_generation", {})
script_parse_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "diagnostic_words_per_minute": int(parser_config.get("diagnostic_words_per_minute", 140)),
        "title_card_seconds": float(parser_config.get("title_card_seconds", 2.8)),
    },
    "input": {
        "response_text": response_text,
        "video_packaging": video_packaging_result,
        "script_source": str(SCRIPT_RESPONSE_PATH),
    },
}
script_generation_result = parse_chatgpt_script(script_parse_input)
if script_generation_result["status"] != "ok":
    raise RuntimeError(script_generation_result["summary"])
script_package = script_generation_result["data"]["script_package"]
write_json(SCRIPT_OUTPUT_DIR / "chatgpt_script_parse_input.json", script_parse_input)
write_json(SCRIPT_RESULT_PATH, script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
FULL_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
FULL_SCRIPT_PATH.write_text(script_package["full_script"] + "\n", encoding="utf-8")
print(f"Canonical result: {SCRIPT_RESULT_PATH}")
print(f"Script words:     {script_generation_result['summary']['total_word_count']}")
print(f"Sections:         {script_generation_result['summary']['section_count']}")


## 4. Optional Visual-Planning Handoff

Run voiceover in `production/voiceover_generation.ipynb` first. This step
packages the canonical script, selected records, available visual families,
and exact audio chunks for ChatGPT.


In [ ]:
if not SCRIPT_RESULT_PATH.exists():
    raise FileNotFoundError(f"Canonical script result not found: {SCRIPT_RESULT_PATH}")
script_generation_result = load_json(SCRIPT_RESULT_PATH)
full_script = script_generation_result["data"]["script_package"]["full_script"]
voiceover_path, voiceover_package = matching_voiceover_package(script_generation_result)
planner_config = render_config_templates(deepcopy(PROFILE_CONFIG["visuals"]["planner"]))

visual_handoff_packet = {
    "package_type": "chatgpt_visual_scene_plan_packet",
    "selected_format": video_packet.get("selected_format", "roundup"),
    "run_id": RUN_ID,
    "script": {
        "script_source": str(SCRIPT_RESULT_PATH),
        "word_count": len(full_script.split()),
        "sections": script_generation_result["data"]["script_package"].get("sections", []),
        "story_asset_hints": script_generation_result["data"]["script_package"].get("story_asset_hints", []),
        "provenance": script_generation_result["data"]["script_package"]["provenance"],
        "full_text": full_script,
    },
    "voiceover_timing": compact_voiceover_timing(voiceover_package),
    "video_packet": video_packet,
    "visual_context": {
        "scene_policy": planner_config.get("scene_policy", {}),
        "visual_capabilities": planner_config.get("visual_capabilities", {}),
        "style": planner_config.get("style", {}),
    },
}
visual_handoff_dir = HANDOFF_DIR / "visuals" / "packet"
visual_handoff_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {"profile": "visual_scene_plan", "prompt_version": "v1", "output_dir": str(visual_handoff_dir)},
    "input": {"manifest": RUN_CONTEXT, "artifacts": {"visual_scene_plan": visual_handoff_packet}, "summaries": {}},
}
visual_handoff_result = run_chatgpt_exporter(visual_handoff_input)
if visual_handoff_result["status"] != "ok":
    raise RuntimeError(visual_handoff_result["summary"])
paths = write_handoff_files(visual_handoff_result["data"]["handoff_package"]["files"], visual_handoff_dir)
write_json(HANDOFF_DIR / "visuals" / "handoff_input.json", visual_handoff_input)
write_json(HANDOFF_DIR / "visuals" / "handoff_result.json", visual_handoff_result)
write_json(visual_handoff_dir / "visual_handoff_packet.json", visual_handoff_packet)
VISUAL_RESPONSE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not VISUAL_RESPONSE_PATH.exists():
    VISUAL_RESPONSE_PATH.write_text("Paste the full CHATGPT_VISUAL_PLAN_RESPONSE_V1 response here, then run section 5.\n", encoding="utf-8")
for path in paths:
    print(path)
print(f"Voiceover source: {voiceover_path}")
print(f"Response file:    {VISUAL_RESPONSE_PATH}")


## 5. Parse Visual-Planning Response


In [ ]:
from src.pipeline.handoffs.chatgpt_visual_plan_parser import run as parse_chatgpt_visual_plan

script_generation_result = load_json(SCRIPT_RESULT_PATH)
full_script = script_generation_result["data"]["script_package"]["full_script"]
_, voiceover_package = matching_voiceover_package(script_generation_result)
planner_config = render_config_templates(deepcopy(PROFILE_CONFIG["visuals"]["planner"]))
response_text = VISUAL_RESPONSE_PATH.read_text(encoding="utf-8")
visual_parse_input = {
    "run": {"run_id": RUN_ID, "generated_at": datetime.now(IST).isoformat()},
    "config": {
        "scene_policy": planner_config.get("scene_policy", {}),
        "visual_capabilities": planner_config.get("visual_capabilities", {}),
        "require_complete_timeline": True,
        "gap_tolerance_seconds": 0.5,
        "output_dir": str(VISUAL_PLAN_OUTPUT_DIR),
    },
    "input": {
        "response_text": response_text,
        "video_packaging": video_packaging_result,
        "script_generation": script_generation_result,
        "script_source": str(SCRIPT_RESULT_PATH),
        "voiceover": voiceover_package,
    },
}
visual_scene_planning_result = parse_chatgpt_visual_plan(visual_parse_input)
if visual_scene_planning_result["status"] != "ok":
    raise RuntimeError(visual_scene_planning_result["summary"].get("failure_reason", "visual plan parse failed"))
visual_plan = visual_scene_planning_result["data"]["visual_scene_plan"]
write_json(VISUAL_PLAN_OUTPUT_DIR / "chatgpt_visual_plan_parse_input.json", visual_parse_input)
write_json(VISUAL_PLAN_OUTPUT_DIR / "visual_scene_planning_result.json", visual_scene_planning_result)
write_json(VISUAL_PLAN_OUTPUT_DIR / "visual_scene_plan.json", visual_plan)
print(f"Scenes:    {visual_scene_planning_result['summary']['scene_count']}")
print(f"Plan path: {VISUAL_PLAN_OUTPUT_DIR / 'visual_scene_planning_result.json'}")
